In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"

In [ ]:

def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def get_metrics(y_true, y_pred, y_score):
    
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_score),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def subject_level_predictions(pred_conv):
    
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["score_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["score_1"] >= 0).astype(int)
    return pred_subject

In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))
meta_cols = {"subject_id", "avatar", "label"}
eeg_cols = [c for c in data.columns if c not in meta_cols and c not in text_cols and c not in speech_cols]
feature_cols = text_cols + speech_cols + eeg_cols

required = {"subject_id", "avatar", "label"}
if not required.issubset(data.columns):
    raise ValueError(f"Faltan columnas obligatorias: {required - set(data.columns)}")


df = data.merge(partitions, on=["subject_id", "avatar"], how="inner")

n_before = len(df)
df = df.dropna(subset=feature_cols).copy()
n_removed = n_before - len(df)

print("Filas iniciales con partición:", n_before)
print("Filas eliminadas por no tener alguna modalidad:", n_removed)
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Variables text:", len(text_cols))
print("Variables speech:", len(speech_cols))
print("Variables EEG:", len(eeg_cols))

print("\nSujetos por outer fold después del filtro:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))

print("\nConversaciones disponibles por narrativa:")
display(df["avatar"].value_counts().rename_axis("avatar").to_frame("n_rows"))

Filas iniciales con partición: 600
Filas eliminadas por no tener alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Variables text: 768
Variables speech: 1024
Variables EEG: 27

Sujetos por outer fold después del filtro:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8



Conversaciones disponibles por narrativa:


,n_rows
avatar,
Sad,94
Neutral1,94
Happy,94
Angry,92
Relax,92
Neutral2,92


In [ ]:
# SVM necesita escalado. El escalado se ajusta dentro de cada train/dev, nunca con test.
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(class_weight="balanced", random_state=SEED, cache_size=2000)),
])


param_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    {"svm__kernel": ["rbf"], "svm__C": [0.01, 0.1, 1, 10, 100], "svm__gamma": ["scale", 0.001, 0.01, 0.1]},
]

n_candidates = 6 + 5 * 4
print("Candidatos SVM por búsqueda:", n_candidates)
print("Fits por búsqueda:", n_candidates * N_SPLITS)

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]

Candidatos SVM por búsqueda: 26
Fits por búsqueda: 130


In [ ]:
OUT_DIR = DATA_DIR / "05_results_text_speech_eeg_emotion_wise_svm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

emotion_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []
all_emotion_predictions = []

emotions = sorted(df["avatar"].unique())
print("Búsquedas GridSearchCV esperadas:", len(df["outer_fold"].unique()) * len(emotions))
print("Fits internos esperados:", len(df["outer_fold"].unique()) * len(emotions) * n_candidates * N_SPLITS)

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")
    fold_predictions = []

    for emotion in emotions:
        t0 = time.time()
        emo_df = df[df["avatar"] == emotion].copy()
        dev = emo_df[emo_df["outer_fold"] != fold].reset_index(drop=True)
        test = emo_df[emo_df["outer_fold"] == fold].reset_index(drop=True)

        if len(test) == 0 or dev["label"].nunique() < 2 or test["label"].nunique() < 2:
            print(f"{emotion}: omitido en fold {fold} por falta de datos/clases")
            continue

        X_dev = dev[feature_cols].to_numpy(dtype=np.float32)
        y_dev = dev["label"].to_numpy(dtype=int)
        groups_dev = dev["subject_id"].to_numpy()

        X_test = test[feature_cols].to_numpy(dtype=np.float32)
        y_test = test["label"].to_numpy(dtype=int)

        inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

        grid = GridSearchCV(
            estimator=svm_pipeline,
            param_grid=param_grid,
            scoring=scoring,
            refit="UAcc",
            cv=inner_cv,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(X_dev, y_dev, groups=groups_dev)

        best_idx = grid.best_index_
        model = grid.best_estimator_

        score_1 = model.decision_function(X_test)
        pred = model.predict(X_test)

        pred_emo = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
        pred_emo["score_1"] = score_1
        pred_emo["pred"] = pred
        fold_predictions.append(pred_emo)
        all_emotion_predictions.append(pred_emo)

        emo_metrics = get_metrics(y_test, pred, score_1)
        emo_metrics["outer_fold"] = fold
        emo_metrics["avatar"] = emotion
        emo_metrics["cv_f1"] = grid.cv_results_["mean_test_f1"][best_idx]
        emotion_metrics_rows.append(emo_metrics)

        best_params_rows.append({
            "outer_fold": fold,
            "avatar": emotion,
            "best_kernel": grid.best_params_["svm__kernel"],
            "best_C": grid.best_params_["svm__C"],
            "best_gamma": grid.best_params_.get("svm__gamma", "not_used"),
            "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
            "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
            "elapsed_seconds": round(time.time() - t0, 1),
        })

        print(f"{emotion}: CV F1={grid.cv_results_['mean_test_f1'][best_idx]:.3f} | Test F1={emo_metrics['f1']:.3f} | tiempo={time.time() - t0:.1f}s")

    if len(fold_predictions) == 0:
        print("No hay predicciones válidas en este fold.")
        continue

    fold_pred = pd.concat(fold_predictions, ignore_index=True)
    pred_subject = subject_level_predictions(fold_pred)
    subject_metrics = get_metrics(pred_subject["label"], pred_subject["pred"], pred_subject["score_1"])
    subject_metrics["outer_fold"] = fold
    subject_metrics["n_subjects"] = pred_subject["subject_id"].nunique()
    subject_metrics_rows.append(subject_metrics)

    print("Subject-level Test F1 agregado:", round(subject_metrics["f1"], 3))

emotion_metrics_df = pd.DataFrame(emotion_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
emotion_predictions_df = pd.concat(all_emotion_predictions, ignore_index=True)
subject_predictions_global = subject_level_predictions(emotion_predictions_df)
global_subject_metrics = get_metrics(
    subject_predictions_global["label"],
    subject_predictions_global["pred"],
    subject_predictions_global["score_1"],
)

subject_summary = pd.DataFrame({
    "metric": ["Subject-level Test F1"],
    "mean": [subject_metrics_df["f1"].mean()],
    "std": [subject_metrics_df["f1"].std()],
}).round(3)

emotion_summary = (
    emotion_metrics_df
    .groupby("avatar")[["cv_f1", "f1", "UAcc", "auc"]]
    .agg(["mean", "std"])
    .round(3)
)

emotion_metrics_df.to_csv(OUT_DIR / "emotion_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold_and_emotion.csv", index=False)
emotion_predictions_df.to_csv(OUT_DIR / "emotion_predictions.csv", index=False)
subject_predictions_global.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)
subject_summary.to_csv(OUT_DIR / "main_subject_level_summary.csv", index=False)

print("\nResumen por narrativa")
display(emotion_summary)

print("\nResultado principal agregado por sujeto")
display(subject_summary)

print("\nMétricas subject-level globales")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("\nArchivos guardados en:", OUT_DIR)

Búsquedas GridSearchCV esperadas: 30
Fits internos esperados: 3900

===== OUTER FOLD 1 =====
Angry: CV F1=0.548 | Test F1=0.571 | tiempo=3.8s
Happy: CV F1=0.516 | Test F1=0.636 | tiempo=2.8s
Neutral1: CV F1=0.500 | Test F1=0.429 | tiempo=2.1s
Neutral2: CV F1=0.471 | Test F1=0.615 | tiempo=1.8s
Relax: CV F1=0.564 | Test F1=0.588 | tiempo=1.8s
Sad: CV F1=0.538 | Test F1=0.667 | tiempo=1.8s
Subject-level Test F1 agregado: 0.706

===== OUTER FOLD 2 =====
Angry: CV F1=0.586 | Test F1=0.500 | tiempo=1.8s
Happy: CV F1=0.559 | Test F1=0.400 | tiempo=2.5s
Neutral1: CV F1=0.604 | Test F1=0.250 | tiempo=2.2s
Neutral2: CV F1=0.490 | Test F1=0.625 | tiempo=1.8s
Relax: CV F1=0.620 | Test F1=0.632 | tiempo=1.7s
Sad: CV F1=0.545 | Test F1=0.000 | tiempo=1.7s
Subject-level Test F1 agregado: 0.5

===== OUTER FOLD 3 =====
Angry: CV F1=0.612 | Test F1=0.714 | tiempo=2.8s
Happy: CV F1=0.530 | Test F1=0.267 | tiempo=2.2s
Neutral1: CV F1=0.439 | Test F1=0.462 | tiempo=2.4s
Neutral2: CV F1=0.524 | Test F1=0.2

cv_f1            f1          UAcc           auc       
           mean    std   mean    std   mean    std   mean    std
avatar                                                          
Angry     0.556  0.094  0.550  0.245  0.641  0.171  0.690  0.169
Happy     0.510  0.040  0.460  0.139  0.535  0.106  0.583  0.134
Neutral1  0.471  0.084  0.359  0.149  0.498  0.136  0.523  0.196
Neutral2  0.485  0.037  0.479  0.261  0.627  0.162  0.604  0.142
Relax     0.560  0.036  0.564  0.083  0.635  0.053  0.691  0.088
Sad       0.492  0.055  0.515  0.307  0.633  0.164  0.652  0.170


Resultado principal agregado por sujeto


,metric,mean,std
0,Subject-level Test F1,0.493,0.228



Métricas subject-level globales


,global
WAcc,0.649
UAcc,0.622
auc,0.660
f1,0.522
precision,0.600
recall,0.462
kappa,0.252



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/05_results_text_speech_eeg_emotion_wise_svm_corrected
